UJI NORMALITAS

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# =============================
# INPUT DATA
# =============================
df = pd.read_csv('Hasil Rasio Keuangan.csv')

rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']

for rasio in rasio_list:
    df[rasio] = df[rasio].astype(str).str.replace(',', '.').astype(float)

df = df[df['Tahun'] >= 2021].reset_index(drop=True)

# =============================
# NILAI KRITIS KOLMOGOROV-SMIRNOV STANDAR D_(n; alpha)
# =============================
N_KUARTAL = 4
ALPHA = 0.05
D_KRITIS = 0.624  # D_(4; 0,05)
TAHUN_LIST = [2021, 2022, 2023, 2024, 2025]


def hitung_ks(data):
    """Menghitung statistik Dn Kolmogorov-Smirnov secara manual, mengikuti
    langkah pada perhitungan manual (persamaan 2.12-2.16): mu_hat, sigma_hat,
    data diurutkan, Fn(x-), Fn(x), F0(x), lalu Dn = maksimum selisihnya."""
    n = len(data)
    mu = np.mean(data)
    sigma = np.std(data, ddof=1)
    sorted_data = np.sort(data)

    baris = []
    Dn = 0.0
    for i, x in enumerate(sorted_data, start=1):
        Fn_minus = (i - 1) / n
        Fn = i / n
        F0 = stats.norm.cdf((x - mu) / sigma)
        d1 = abs(Fn - F0)
        d2 = abs(F0 - Fn_minus)
        Dn = max(Dn, d1, d2)
        baris.append((i, x, Fn_minus, Fn, F0, d1, d2))

    return mu, sigma, baris, Dn


# =============================
# PERHITUNGAN LENGKAP UNTUK SELURUH KELOMPOK TAHUN
# (2021-2025) PADA KEDELAPAN RASIO KEUANGAN
# =============================
hasil_ks = []

for rasio in rasio_list:
    if rasio not in df.columns:
        print(f"Peringatan: Kolom '{rasio}' tidak ditemukan di CSV Anda!")
        continue

    print("\n" + "#" * 78)
    print(f"#  RASIO: {rasio}")
    print("#" * 78)

    for tahun in TAHUN_LIST:
        data = df[df['Tahun'] == tahun][rasio].dropna().values

        if len(data) != N_KUARTAL:
            print(f"Peringatan: kelompok tahun {tahun} untuk {rasio} "
                  f"tidak berisi {N_KUARTAL} observasi (ditemukan {len(data)}).")
            continue

        mu, sigma, baris, Dn = hitung_ks(data)

        print(f"\n{'='*78}")
        print(f"  {rasio} — KELOMPOK TAHUN {tahun} (n = 4)")
        print(f"{'='*78}")
        print(f"Data: {[round(float(v), 4) for v in data]}")
        print(f"mu_hat    = {mu:.4f}")
        print(f"sigma_hat = {sigma:.4f}\n")

        print(f"{'i':<3}{'x_i(urut)':>12}{'Fn(x-)':>10}{'Fn(x)':>10}{'F0(x)':>10}"
              f"{'|Fn-F0|':>12}{'|F0-Fn-|':>12}")
        for i, x, Fn_minus, Fn, F0, d1, d2 in baris:
            print(f"{i:<3}{x:>12.4f}{Fn_minus:>10.4f}{Fn:>10.4f}{F0:>10.4f}"
                  f"{d1:>12.4f}{d2:>12.4f}")

        keputusan = 'Tolak H0' if Dn >= D_KRITIS else 'Gagal Tolak H0'
        kesimpulan = 'Tidak Normal' if Dn >= D_KRITIS else 'Normal'

        print(f"\nDn = {Dn:.4f}")
        print(f"D_(4;0,05) kritis = {D_KRITIS}")
        print(f"Karena Dn ({Dn:.4f}) {'<' if kesimpulan=='Normal' else '>='} "
              f"D_(4;0,05) ({D_KRITIS}), maka H0 "
              f"{'gagal ditolak' if kesimpulan=='Normal' else 'ditolak'}, "
              f"data {rasio} kelompok {tahun} dinyatakan berdistribusi "
              f"{kesimpulan.lower()}.")

        hasil_ks.append({
            'Rasio': rasio,
            'Tahun': tahun,
            'Statistik Dn': round(Dn, 4),
            f'D_({N_KUARTAL},{ALPHA}) kritis': D_KRITIS,
            'Keputusan': keputusan,
            'Kesimpulan': kesimpulan
        })

df_ks = pd.DataFrame(hasil_ks)

# =============================
# TABEL RINGKASAN
# (baris = rasio, kolom = tahun, isi = Dn per kelompok)
# =============================
print("\n" + "=" * 78)
print("  TABEL 4.15 HASIL UJI NORMALITAS (KOLMOGOROV-SMIRNOV) PER KELOMPOK TAHUN")
print("=" * 78)

pivot = df_ks.pivot(index='Rasio', columns='Tahun', values='Statistik Dn').loc[rasio_list]
kesimpulan_rasio = {}
for rasio in rasio_list:
    sub = df_ks[df_ks['Rasio'] == rasio]
    semua_normal = (sub['Kesimpulan'] == 'Normal').all()
    kesimpulan_rasio[rasio] = 'Normal seluruh kelompok' if semua_normal else 'Ada kelompok tidak normal'
pivot['Kesimpulan'] = pd.Series(kesimpulan_rasio)

print(pivot.to_string())
print(f"\n(nilai adalah statistik Dn tiap kelompok; nilai kritis D_(4,0.05) = "
      f"{D_KRITIS} berlaku untuk seluruh sel)")

# =============================
# RINGKASAN: kelompok tahun yang TIDAK normal (jika ada)
# =============================
print("\n=== RINGKASAN KELOMPOK YANG MENYIMPANG DARI NORMALITAS ===")
tidak_normal = df_ks[df_ks['Kesimpulan'] == 'Tidak Normal']
if tidak_normal.empty:
    print("Tidak ada kelompok tahun yang menyimpang dari normalitas "
          f"(Dn tertinggi = {df_ks['Statistik Dn'].max():.4f}, "
          f"pada {df_ks.loc[df_ks['Statistik Dn'].idxmax(), 'Rasio']} "
          f"kelompok {df_ks.loc[df_ks['Statistik Dn'].idxmax(), 'Tahun']}).")
else:
    for _, row in tidak_normal.iterrows():
        print(f"  {row['Rasio']} - Tahun {row['Tahun']}: Dn = {row['Statistik Dn']:.4f} "
              f"(>= D_({N_KUARTAL},{ALPHA}) = {D_KRITIS}) -> Tidak Normal")


##############################################################################
#  RASIO: ROIC
##############################################################################

  ROIC — KELOMPOK TAHUN 2021 (n = 4)
Data: [1.1196, 1.4313, 1.2385, 0.9239]
mu_hat    = 1.1783
sigma_hat = 0.2128

i     x_i(urut)    Fn(x-)     Fn(x)     F0(x)     |Fn-F0|    |F0-Fn-|
1        0.9239    0.0000    0.2500    0.1159      0.1341      0.1159
2        1.1196    0.2500    0.5000    0.3913      0.1087      0.1413
3        1.2385    0.5000    0.7500    0.6113      0.1387      0.1113
4        1.4313    0.7500    1.0000    0.8828      0.1172      0.1328

Dn = 0.1413
D_(4;0,05) kritis = 0.624
Karena Dn (0.1413) < D_(4;0,05) (0.624), maka H0 gagal ditolak, data ROIC kelompok 2021 dinyatakan berdistribusi normal.

  ROIC — KELOMPOK TAHUN 2022 (n = 4)
Data: [0.8837, 1.4568, -0.0019, 1.2577]
mu_hat    = 0.8991
sigma_hat = 0.6459

i     x_i(urut)    Fn(x-)     Fn(x)     F0(x)     |Fn-F0|    |F0-Fn-|
1       -0.00

UJI HOMOGENITAS VARIANSI (ANOMV)

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist

# =============================
# LOAD DATA
# =============================
df_kuartalan = pd.read_csv('Hasil Rasio Keuangan.csv')

rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']

# Fix format koma ke titik
for rasio in rasio_list:
    if rasio in df_kuartalan.columns:
        df_kuartalan[rasio] = df_kuartalan[rasio].astype(str).str.replace(',', '.').astype(float)

tahun_list = sorted(df_kuartalan['Tahun'].unique().tolist())
n = 4
I = len(tahun_list)
N = I * n

# =============================
# PARAMETER ANOMV
# =============================
vi            = n - 1
ve            = N - I
alpha         = 0.05
alpha_koreksi = alpha / (2 * I)

UDLv = f_dist.ppf(1 - alpha_koreksi, dfn=vi, dfd=ve)
LDLv = 1 / f_dist.ppf(1 - alpha_koreksi, dfn=ve, dfd=vi)

# =============================
# FUNGSI ANOMV
# =============================
def anomv(df_kuartalan, rasio, I, n):
    kelompok = [df_kuartalan[rasio].iloc[i*n:(i+1)*n].values for i in range(I)]
    Si2  = [np.var(k, ddof=1) for k in kelompok]
    SSe  = sum((n - 1) * s for s in Si2)
    MSe  = SSe / (N - I)
    Fi   = [s / MSe for s in Si2]
    return Si2, MSe, Fi

# =============================
# OUTPUT: PER RASIO
# =============================
print("=" * 75)
print("            HASIL UJI HOMOGENITAS VARIANSI (ANOMV)")
print(f"  α = {alpha}  |  α/(2I) = {alpha_koreksi:.5f}  |  "
      f"UDLᵥ = {UDLv:.4f}  |  LDLᵥ = {LDLv:.4f}")
print(f"  vᵢ = {vi}  |  vₑ = {ve}")
print("=" * 75)

mse_per_rasio = {}

for rasio in rasio_list:
    Si2, MSe, Fi = anomv(df_kuartalan, rasio, I, n)
    mse_per_rasio[rasio] = MSe

    print(f"\n┌─ RASIO: {rasio}   MSe = {MSe:.6f}")
    print(f"  {'Tahun':<8} {'Sᵢ²':>14} {'Fᵢ = Sᵢ²/MSe':>16} {'Keputusan':>15}")
    print(f"  {'─'*55}")
    semua_homogen = True
    for tahun, s, f in zip(tahun_list, Si2, Fi):
        status = "Homogen" if LDLv <= f <= UDLv else "TIDAK Homogen"
        if status != "Homogen":
            semua_homogen = False
        print(f"  {tahun:<8} {s:>14.6f} {f:>16.6f} {status:>15}")
    print(f"  {'─'*55}")
    kesimpulan = "✔ Semua kelompok HOMOGEN" if semua_homogen else "✘ Ada kelompok TIDAK HOMOGEN"
    print(f"  Kesimpulan: {kesimpulan}")

# =============================
# RINGKASAN MSE
# =============================
print(f"\n{'=' * 75}")
print("  RINGKASAN MSE PER RASIO ")
print(f"{'=' * 75}")
print(f"  {'Rasio':<8} {'MSE':>14}")
print(f"  {'─'*24}")
for rasio, mse in mse_per_rasio.items():
    print(f"  {rasio:<8} {mse:>14.6f}")

            HASIL UJI HOMOGENITAS VARIANSI (ANOMV)
  α = 0.05  |  α/(2I) = 0.00500  |  UDLᵥ = 6.4760  |  LDLᵥ = 0.0232
  vᵢ = 3  |  vₑ = 15

┌─ RASIO: ROIC   MSe = 0.190380
  Tahun               Sᵢ²     Fᵢ = Sᵢ²/MSe       Keputusan
  ───────────────────────────────────────────────────────
  2021           0.045266         0.237767         Homogen
  2022           0.417220         2.191518         Homogen
  2023           0.021831         0.114672         Homogen
  2024           0.073441         0.385763         Homogen
  2025           0.394139         2.070280         Homogen
  ───────────────────────────────────────────────────────
  Kesimpulan: ✔ Semua kelompok HOMOGEN

┌─ RASIO: ROA   MSe = 0.187351
  Tahun               Sᵢ²     Fᵢ = Sᵢ²/MSe       Keputusan
  ───────────────────────────────────────────────────────
  2021           0.008034         0.042884         Homogen
  2022           0.083846         0.447537         Homogen
  2023           0.028515         0.152204         

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# LOAD DATA
# =============================
df_kuartalan = pd.read_csv('Hasil Rasio Keuangan.csv')

rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']

for rasio in rasio_list:
    if rasio in df_kuartalan.columns:
        df_kuartalan[rasio] = df_kuartalan[rasio].astype(str).str.replace(',', '.').astype(float)

tahun_list = sorted(df_kuartalan['Tahun'].unique().tolist())
n = 4
I = len(tahun_list)
N = I * n

# =============================
# PARAMETER ANOM
# =============================
h = 2.88

# =============================
# FUNGSI ANOM
# =============================
def hitung_anom(df_kuartalan, rasio, MSE, I, n, N, h):
    """Menghitung grand mean, rata-rata kelompok, Di, dan decision limits ANOM
    (persamaan 2.25-2.33)."""

    grand_mean = df_kuartalan[rasio].sum() / N

    yi_bar = np.array([
        df_kuartalan[rasio].iloc[i * n:(i + 1) * n].mean() for i in range(I)
    ])

    Di = yi_bar - grand_mean

    margin = h * np.sqrt(MSE * (I - 1) / N)
    UDL = grand_mean + margin
    LDL = grand_mean - margin

    return yi_bar, grand_mean, Di, UDL, LDL


# =============================
# DECISION CHART
# =============================
def plot_decision_chart(rasio, tahun_list, yi_bar, GM, UDL, LDL, alpha=0.05, save=True):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    x = np.arange(1, len(tahun_list) + 1)

    ax.axhline(UDL, color='firebrick', linestyle='--', linewidth=1.4, zorder=2,
               label=f'UDL = {UDL:.4f}')
    ax.axhline(GM,  color='seagreen',  linestyle='-',  linewidth=1.4, zorder=2,
               label=f'Grand Mean (CL) = {GM:.4f}')
    ax.axhline(LDL, color='royalblue', linestyle='--', linewidth=1.4, zorder=2,
               label=f'LDL = {LDL:.4f}')

    out_mask = (yi_bar > UDL) | (yi_bar < LDL)
    for xi, yi, out in zip(x, yi_bar, out_mask):
        ax.vlines(xi, GM, yi, color='steelblue', linewidth=1.2, zorder=3)
        if out:
            ax.plot(xi, yi, marker='s', color='firebrick', markersize=9,
                    markeredgecolor='firebrick', zorder=5)
        else:
            ax.plot(xi, yi, marker='o', color='steelblue', markersize=8,
                    markerfacecolor='white', markeredgewidth=1.6, zorder=5)
        offset = 10 if yi >= GM else -14
        va = 'bottom' if yi >= GM else 'top'
        ax.annotate(f'{yi:.4f}', xy=(xi, yi), xytext=(0, offset),
                    textcoords='offset points', ha='center', va=va,
                    fontsize=8.5, fontweight='bold' if out else 'normal',
                    color='firebrick' if out else 'black')

    ax.set_title(f'Decision Chart ANOM — {rasio}', fontsize=13, fontweight='bold', pad=28)
    ax.text(0.5, 1.03, f'α = {alpha}', transform=ax.transAxes,
            ha='center', fontsize=10, color='dimgray')
    ax.set_xlabel('Tahun', fontsize=11)
    ax.set_ylabel(f'Rata-rata {rasio}', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(tahun_list)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
              fontsize=9, frameon=False)

    y_all = np.concatenate([yi_bar, [UDL, LDL]])
    pad = (y_all.max() - y_all.min()) * 0.15
    ax.set_ylim(y_all.min() - pad, y_all.max() + pad)
    plt.tight_layout()
    if save:
        plt.savefig(f'decision_chart_{rasio}.png', dpi=150, bbox_inches='tight')
    return fig


# =============================
# OUTPUT TABEL + DECISION CHART PER RASIO
# =============================
print("=" * 70)
print("           HASIL ANALISIS ANOM KINERJA KEUANGAN")
print(f"  h = {h}  |  α = 0.05  |  I = {I}  |  n = {n}  |  N = {N}  |  ν = {N-I}")
print("  Rumus UDL/LDL: ȳ.. ± h · √(MSE · (I-1)/N)  [Persamaan 2.32-2.33]")
print("=" * 70)

ringkasan = []
for rasio in rasio_list:

    MSE = mse_per_rasio[rasio]

    if np.isnan(MSE):
        print(f"\nSkipping ANOM for RASIO: {rasio} due to NaN MSE.\n")
        ringkasan.append((rasio, np.nan, np.nan, np.nan, False))
        continue

    yi_bar, GM, Di, UDL, LDL = hitung_anom(df_kuartalan, rasio, MSE, I, n, N, h)

    print(f"\n┌─ RASIO: {rasio}")
    print(f"  Grand Mean (ȳ..) = {GM:.4f}  |  UDL = {UDL:.4f}  |  LDL = {LDL:.4f}")
    print(f"  {'Tahun':<8} {'ȳᵢ':>10} {'Dᵢ = ȳᵢ - ȳ..':>16} {'Keputusan':>18}")
    print(f"  {'─'*54}")

    ada_anomali = False
    for i, tahun in enumerate(tahun_list):
        if yi_bar[i] > UDL:
            status = "Signifikan tinggi"
            ada_anomali = True
        elif yi_bar[i] < LDL:
            status = "Signifikan rendah"
            ada_anomali = True
        else:
            status = "Dalam batas"
        print(f"  {tahun:<8} {yi_bar[i]:>10.4f} {Di[i]:>16.4f} {status:>18}")

    print(f"  {'─'*54}")

    if ada_anomali:
        kesimpulan = "✘ Ada kelompok yang menyimpang signifikan"
    else:
        kesimpulan = "✔ Tidak ada kelompok yang menyimpang signifikan"
    print(f"  Kesimpulan: {kesimpulan}")
    ringkasan.append((rasio, GM, UDL, LDL, ada_anomali))

    _ = plot_decision_chart(rasio, tahun_list, yi_bar, GM, UDL, LDL)
    plt.close(_)

# =============================
# RINGKASAN AKHIR
# =============================
print(f"\n{'=' * 70}")
print("           RINGKASAN AKHIR HASIL ANOM")
print(f"{'=' * 70}")
print(f"  {'Rasio':<8} {'Grand Mean':>12} {'UDL':>12} {'LDL':>12} {'Anomali':>10}")
print(f"  {'─'*60}")
for rasio, GM, UDL, LDL, anomali in ringkasan:
    anomali_str = "Ya" if anomali else "Tidak"
    print(f"  {rasio:<8} {GM:>12.4f} {UDL:>12.4f} {LDL:>12.4f} {anomali_str:>10}")
print(f"  {'─'*60}")

           HASIL ANALISIS ANOM KINERJA KEUANGAN
  h = 2.88  |  α = 0.05  |  I = 5  |  n = 4  |  N = 20  |  ν = 15
  Rumus UDL/LDL: ȳ.. ± h · √(MSE · (I-1)/N)  [Persamaan 2.32-2.33]

┌─ RASIO: ROIC
  Grand Mean (ȳ..) = 1.0921  |  UDL = 1.6541  |  LDL = 0.5301
  Tahun            ȳᵢ    Dᵢ = ȳᵢ - ȳ..          Keputusan
  ──────────────────────────────────────────────────────
  2021         1.1783           0.0862        Dalam batas
  2022         0.8991          -0.1930        Dalam batas
  2023         1.2089           0.1168        Dalam batas
  2024         1.2933           0.2011        Dalam batas
  2025         0.8810          -0.2111        Dalam batas
  ──────────────────────────────────────────────────────
  Kesimpulan: ✔ Tidak ada kelompok yang menyimpang signifikan

┌─ RASIO: ROA
  Grand Mean (ȳ..) = 0.3098  |  UDL = 0.8673  |  LDL = -0.2477
  Tahun            ȳᵢ    Dᵢ = ȳᵢ - ȳ..          Keputusan
  ──────────────────────────────────────────────────────
  2021         0.4708   